## Decision Trees Implementation

In [1]:
# importing the libraries and dataset
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
# entropy
def calc_entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    return -np.sum(proportions * np.log2(proportions))

# information gain
def information_gain(X_column, y, threshold):

    parent_entropy = calc_entropy(y)

    left_mask = X_column <= threshold
    right_mask = X_column > threshold
    y_left, y_right = y[left_mask], y[right_mask]

    if len(y_left) == 0 or len(y_right) == 0:
        return 0

    n = len(y)
    weighted_entropy = (len(y_left)/n) * calc_entropy(y_left) + \
                       (len(y_right)/n) * calc_entropy(y_right)

    return parent_entropy - weighted_entropy

# best split
def best_split(X, y):
    best_gain = 0
    best_feature = None
    best_threshold = None

    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            gain = information_gain(X[:, feature], y, threshold)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold

# tree node
class Node:
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction

# building the tree
def build_tree(X, y, max_depth=10, depth=0):

    if len(np.unique(y)) == 1:
        return Node(prediction=y[0])
    if depth >= max_depth:
        return Node(prediction=np.bincount(y).argmax())
    if len(y) < 2:
        return Node(prediction=np.bincount(y).argmax())

    feature, threshold = best_split(X, y)

    if feature is None:
        return Node(prediction=np.bincount(y).argmax())

    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    left = build_tree(X[left_mask], y[left_mask], max_depth, depth+1)
    right = build_tree(X[right_mask], y[right_mask], max_depth, depth+1)

    return Node(feature=feature, threshold=threshold, left=left, right=right)

# prediction
def predict_one(node, x):
    if node.prediction is not None:
        return node.prediction
    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(node, X):
    return np.array([predict_one(node, x) for x in X])

# training and evaluation
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

tree = build_tree(X_train, y_train, max_depth=3)

predictions = predict(tree, X_test)
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")



Accuracy: 100.00%


In [3]:
# depth-10 tree
for depth in range(1, 11):
    tree = build_tree(X_train, y_train, max_depth=depth)
    train_acc = np.mean(predict(tree, X_train) == y_train)
    test_acc = np.mean(predict(tree, X_test) == y_test)
    print(f"Depth {depth:2d} | Train: {train_acc*100:.1f}% | Test: {test_acc*100:.1f}%")

Depth  1 | Train: 67.5% | Test: 63.3%
Depth  2 | Train: 95.0% | Test: 96.7%
Depth  3 | Train: 95.8% | Test: 100.0%
Depth  4 | Train: 97.5% | Test: 100.0%
Depth  5 | Train: 99.2% | Test: 100.0%
Depth  6 | Train: 100.0% | Test: 100.0%
Depth  7 | Train: 100.0% | Test: 100.0%
Depth  8 | Train: 100.0% | Test: 100.0%
Depth  9 | Train: 100.0% | Test: 100.0%
Depth 10 | Train: 100.0% | Test: 100.0%
